# Chapitre 6 - Lab : Redis Cluster, un partitionnement par hachage grandeur nature

Ce lab confronte le cours à un vrai système : **Redis Cluster** (Redis 7). Son schéma
est exactement l'un de ceux du chapitre : l'espace des clés est découpé une fois pour
toutes en **16 384 slots** (un nombre fixe de partitions), chaque
clé tombe dans le slot `CRC16(clé) mod 16384`, et chaque nœud maître possède un
ensemble de slots. Le routage est confié au **client** : un nœud qui reçoit une clé
qu'il ne possède pas répond par une redirection (`MOVED`), à charge pour le client de
tenir sa carte à jour. Nous allons :

1. créer un cluster de trois maîtres et lire sa carte des slots ;
2. vérifier notre propre CRC16 contre celui du serveur et mesurer la répartition des clés ;
3. observer les redirections `MOVED` et comparer deux façons de tenir la carte à jour ;
4. regrouper des clés avec les **hash tags**, et en voir le revers (le point chaud) ;
5. ajouter un quatrième nœud et **mesurer** la fraction de clés déplacées ;
6. migrer un slot à la main pour voir la **période de bascule** (redirections `ASK`) ;
7. observer une clé chaude depuis les compteurs des nœuds, puis la saler.

**Démarrer et arrêter les services** (depuis la racine du dépôt) :

```bash
docker compose -f docker/partitioning/compose.yml -p ddia_partitioning up -d     # démarrer
docker compose -f docker/partitioning/compose.yml -p ddia_partitioning down -v   # arrêter et effacer
```

Les quatre nœuds Redis partagent le réseau de la machine : ils écoutent sur les ports
**10701 à 10704** (bus de cluster sur 10711 à 10714) et s'annoncent sous l'adresse
`127.0.0.1`, si bien que les redirections renvoient vers des adresses joignables depuis
ce notebook. Le notebook pilote `redis-cli` dans un conteneur via `docker exec`, et se
rejoue autant de fois qu'on veut : il remet les nœuds à zéro avant de créer le cluster.

In [1]:
import subprocess
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import redis
from redis.cluster import RedisCluster
from redis.exceptions import AskError, MovedError, ResponseError

from ddia import partitioning as P
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, COLORS, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 100)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")      # 1234567 -> « 1 234 567 »

PORTS = [10701, 10702, 10703, 10704]
NODE_COLOR = {10701: INDIGO, 10702: TEAL, 10703: AMBER, 10704: ROSE}
N_SLOTS = 16384


def node(port):
    """Connexion simple (non « cluster ») à un nœud."""
    return redis.Redis(host="127.0.0.1", port=port, decode_responses=True)


def cli(*args, check=True):
    """Lance redis-cli dans le conteneur du premier nœud (réseau de l'hôte)."""
    r = subprocess.run(["docker", "exec", "ddia_partitioning_r1", "redis-cli", *args],
                       capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(r.stdout[-500:] + r.stderr[-500:])
    return r.stdout


for port in PORTS:
    for essai in range(30):
        try:
            assert node(port).ping()
            break
        except redis.ConnectionError:
            time.sleep(1)
    else:
        raise RuntimeError(f"le nœud {port} ne répond pas : avez-vous lancé docker compose up ?")
print(node(10701).info("server")["redis_version"], "sur les ports", PORTS)

7.4.11 sur les ports [10701, 10702, 10703, 10704]


**Création du cluster.** On remet les quatre nœuds à zéro (`FLUSHALL` puis
`CLUSTER RESET HARD`, qui leur fait oublier tout cluster précédent), puis `redis-cli
--cluster create` répartit les 16 384 slots entre les **trois premiers** nœuds, sans
répliques (la réplication est le sujet du chapitre 5 ; on l'ignore ici
comme le fait le chapitre). Le quatrième nœud reste à l'écart : il servira au
rééquilibrage de la section 5. Les nœuds apprennent l'existence les uns des autres par
un protocole de **gossip** sur le bus de cluster : il faut attendre qu'ils soient tous
d'accord avant de travailler.

In [2]:
def wait_cluster(ports, timeout=60):
    """Attend que chaque nœud connaisse tous les autres et déclare le cluster opérationnel."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        infos = [node(p).cluster("info") for p in ports]
        if all(i["cluster_state"] == "ok" and int(i["cluster_known_nodes"]) == len(ports) for i in infos):
            return time.time() - t0
        time.sleep(0.2)
    raise TimeoutError("le cluster ne converge pas")


for port in PORTS:
    n = node(port)
    n.flushall()
    n.execute_command("CLUSTER", "RESET", "HARD")

out = cli("--cluster", "create", *[f"127.0.0.1:{p}" for p in PORTS[:3]],
          "--cluster-replicas", "0", "--cluster-yes")
print("\n".join(l for l in out.splitlines() if l.startswith("[OK]")))
print(f"convergence du gossip : {wait_cluster(PORTS[:3]):.1f} s")

[OK] All nodes agree about slots configuration.
[OK] All 16384 slots covered.


convergence du gossip : 4.0 s
